In [1]:
import filetype

In [3]:
import asyncio 
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from openinference.instrumentation.pydantic_ai import OpenInferenceSpanProcessor
from pydantic_ai import Agent
from pydantic_ai.models.test import TestModel
from cc_copilot.agents.chat_review_agent import review_chat
from cc_copilot.types.model_choice import ModelChoice

In [7]:
exporter = InMemorySpanExporter()
provider = TracerProvider()
trace.set_tracer_provider(provider)
provider 

In [9]:
provider.add_span_processor(OpenInferenceSpanProcessor())
provider.add_span_processor(SimpleSpanProcessor(exporter))

In [17]:
async def run():
    await review_chat(ModelChoice(model=TestModel()), "Agent: هلا، هويتك 1000000008")
await run()
print("WITHOUT Agent.instrument_all():", len(exporter.get_finished_spans()), "spans")


WITHOUT Agent.instrument_all(): 0 spans


/Users/xina/Downloads/contact-center-copilot/.venv/lib/python3.15/site-packages/genai_prices/data.py:16293: RuntimeWarning: coroutine 'run' was never awaited
  prices=ModelPrice(input_mtok=Decimal('10'), output_mtok=Decimal('30')),
                 pydantic-ai v2.57.0 • Python 3.15.2

      / \        agent: chat_review_agent • model: test:test • output: ChatReviewOutput • tools: 0
     /   \         capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Logfire and a GitHub login: https://pydantic.dev/ai-setup.md
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


In [20]:
Agent.instrument_all()
await run()
print("WITH Agent.instrument_all():", len(exporter.get_finished_spans()), "spans")

WITH Agent.instrument_all(): 2 spans


In [22]:
spans = exporter.get_finished_spans()
for s in spans:
    print("  span:", s.name, "| kind attr:", s.attributes.get("openinference.span.kind"))

  span: chat test | kind attr: LLM
  span: invoke_agent chat_review_agent | kind attr: AGENT


In [23]:
spans 

(<opentelemetry.sdk.trace.ReadableSpan at 0x1110b8ec0>,
 <opentelemetry.sdk.trace.ReadableSpan at 0x1110b5f90>)

In [27]:
for s in spans:
    print("  span:", s.name, 
    "| kind attr:", s.attributes.get("openinference.span.kind"),
    " | parent span:", s.parent.span_id if s.parent else None ,
    " | trace_id:", s.context.trace_id ,
    " | span_id:", s.context.span_id,
    " | status:", s.status.status_code,
    " | start_time:", s.start_time,
    " | end_time:", s.end_time,
    " | duration:", s.end_time - s.start_time,
    " | attributes:", s.attributes,
    " | events:", s.events,
    " | links:", s.links,
    " | resource:", s.resource,
    " | instrumentation_scope:", s.instrumentation_scope,
    " | kind:", s.kind,
    " | trace_state:", s.context.trace_state,
    " | trace_flags:", s.context.trace_flags,
    " | trace_id_hex:", format(s.context.trace_id, '032x'),
    " | span_id_hex:", format(s.context.span_id, '016x'),
    " | parent_span_id_hex:", format(s.parent.span_id, '016x') if s.parent else None,
    " | status_code:", s.status.status_code,
    " | status_description:", s.status.description)

  span: chat test | kind attr: LLM  | parent span: 8405979733677055740  | trace_id: 4152809034797926068134073107341052219  | span_id: 9135836447312353405  | status: StatusCode.OK  | start_time: 1793963933750829000  | end_time: 1793963933767543000  | duration: 16714000  | attributes: {'gen_ai.provider.name': 'test', 'gen_ai.system': 'test', 'gen_ai.request.model': 'test', 'model_request_parameters': '{"function_tools":[],"native_tools":[],"tool_visibility":{},"revealed_tool_names":[],"deferred_capability_ids":[],"output_mode":"tool","output_object":null,"output_tools":[{"name":"final_result","parameters_json_schema":{"properties":{"rationale":{"description":"Explanation of the assessment, quoting the phrases that triggered each flag","type":"string"},"dialect":{"description":"Main dialect or language the agent used","enum":["najdi","hijazi","gulf","msa","english","arabizi","mixed","other"],"type":"string"},"is_unfriendly":{"description":"Whether the agent\'s tone is unfriendly","type":"

In [ ]:
import os 
from contextlib import asynccontextmanager 
from functools import lru_cache

from fastapi import Depends, FastAPI, File , HTTPException, UploadFile


In [29]:
from cc_copilot.pii.saudi_patterns import mask_pii

In [30]:
masked , matches = mask_pii("هلا، هويتك 1000000008")

In [31]:
masked 

'هلا، هويتك [NATIONAL_ID]'

In [32]:
matches

[PIIMatch(type=<PIIType.NATIONAL_ID: 'NATIONAL_ID'>, start=11, end=21, value='1000000008')]